# Notebook 5 — DinoTXT Zero-Shot Verb Prediction

This notebook performs zero-shot verb prediction on YouCook2 using the
precomputed DinoTXT video embeddings from NB4.

**Approach:**
- Load precomputed video embedding for each segment (`{segment_id}.pt`)
- For each segment, fill the `[MASK]` slot in the masked caption with
  every verb in the vocabulary to produce candidate text descriptions
- Encode all candidates with the frozen DinoTXT text tower
- Pick the verb whose text embedding is most similar to the video embedding
- No training — purely zero-shot similarity in the shared vision-language space

**2026-08-26 fix:** the main loop previously iterated over the full 13,003-segment
dataframe (train+val combined), which is why it estimated ~6h and was interrupted
at 566/13,003 (only 134 landing in the 3,268-segment val split). It now iterates
over `df_val` only (3,268 segments, ~1.5-2h), which is both faster and the only
subset the headline number actually needs — matching what NB6 (V-JEPA2) evaluates
on. A standalone equivalent, `notebooks/scripts/dinov2_zeroshot_fullval_fixed.py`,
already ran this to completion in the background; if you rerun this notebook it
should reproduce the same `processed/dinov2_zeroshot_fullval_fixed.csv`.

**Reads:**
- `annotations/segments_verbs_timed_clean.csv` — captions, verbs, segment IDs (13,003 rows, 9,735 train / 3,268 val)
- `embeddings/dinov2/{segment_id}.pt` — precomputed video embeddings from NB4

**Produces:**
- Top-1 and Top-5 verb prediction accuracy on the full validation set
- `processed/dinov2_zeroshot_fullval_fixed.csv`

## Setup

## Load data

In [ ]:
import os, sys, json
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

In [ ]:
df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv')
# Build verb vocabulary from all extracted verbs
verb_vocab = sorted(df['extracted_verb'].unique().tolist())
n_verbs    = len(verb_vocab)

df_val = df[df['subset'] == 'validation'].reset_index(drop=True)

print(f"Total segments : {len(df):,}")
print(f"Val segments   : {len(df_val):,}")
print(f"Verb classes   : {n_verbs}")
print(f"Columns        : {df.columns.tolist()}")
print(df.head(2))

## Load DinoTXT model (frozen)

In [ ]:
DINOV2_REPO = os.path.join(PATHS['repos'], 'dinov2')
if DINOV2_REPO not in sys.path:
    sys.path.insert(0, DINOV2_REPO)

from dinov2.hub.dinotxt import dinov2_vitl14_reg4_dinotxt_tet1280d20h24l, get_tokenizer

print("Loading DinoTXT...")
dinotxt   = dinov2_vitl14_reg4_dinotxt_tet1280d20h24l().to(DEVICE)
tokenizer = get_tokenizer()
dinotxt.eval()

for p in dinotxt.parameters():
    p.requires_grad = False

total     = sum(p.numel() for p in dinotxt.parameters())
trainable = sum(p.numel() for p in dinotxt.parameters() if p.requires_grad)
print(f"Model loaded: {total/1e6:.1f}M parameters, {trainable} trainable (must be 0)")
assert trainable == 0, "Model must be fully frozen"
print("All parameters frozen ✓")

## Precompute verb text embeddings

We encode each verb once per unique caption context.
For each segment, we fill `[MASK]` with every verb in the vocabulary
using that segment's actual masked caption — e.g.:
  `'[MASK] the onions into small pieces'`
becomes:
  `['chop the onions into small pieces', 'stir the onions ...', ...]`

This uses the real caption context rather than a generic template,
which gives the text tower more signal to work with.

In [ ]:
print(dir(dinotxt.text_model.backbone))

In [ ]:
CONTEXT_LENGTH = dinotxt.text_model.backbone.positional_embedding.shape[0]
print(f"Context length from positional_embedding: {CONTEXT_LENGTH}")

@torch.no_grad()
def get_text_embeddings_for_segment(masked_caption: str) -> torch.Tensor:
    """
    Given a masked caption like '[MASK] the onions into small pieces',
    replace [MASK] with every verb in vocab and encode all candidates.
    Returns tensor of shape (n_verbs, SHARED_DIM), L2-normalised.
    """
    caption_context = masked_caption.replace('[MASK]', '{}')
    verb_texts      = [caption_context.format(v) for v in verb_vocab]

    # tokenize in one batch
    tokens        = tokenizer.tokenize(verb_texts).to(DEVICE)
    text_features = dinotxt.encode_text(tokens)           # (n_verbs, SHARED_DIM)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


# Quick sanity check
test_caption = df['masked_caption'].iloc[0]
test_emb     = get_text_embeddings_for_segment(test_caption)
print(f"Test caption          : {test_caption}")
print(f"Text embeddings shape : {test_emb.shape}  (expected: ({n_verbs}, SHARED_DIM))")
print(f"L2 norm               : {test_emb.norm(dim=-1).mean():.4f}  (expected: ~1.0)")

## Zero-shot verb prediction

For each segment:
1. Load the precomputed video embedding from NB4
2. Encode all verb candidates in the caption context
3. Pick the verb with the highest cosine similarity to the video embedding
4. Record Top-1 and Top-5 correctness

In [ ]:
import json, os
from tqdm import tqdm  # instead of tqdm.notebook

EMB_DIR   = PATHS['embeddings']
# Fixed: was "results_checkpoint.jsonl" in the working dir, which mixes train+val
# rows from the old interrupted run. New file, val-only, in processed/ so it's not
# confused with the stale one.
CKPT_PATH = os.path.join(PATHS['processed'], 'dinov2_zeroshot_fullval_checkpoint.jsonl')

# ── Resume from checkpoint ──────────────────────────────────────────
done_ids = set()
results  = []

if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            row = json.loads(line)
            results.append(row)
            done_ids.add(row['segment_id'])
    print(f"Resumed from checkpoint: {len(results):,} rows already done")

# ── Main loop — validation set only (was the full train+val df) ─────
missing = 0

with open(CKPT_PATH, "a") as ckpt_file:
    for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Zero-shot prediction (full val)"):

        if row['segment_id'] in done_ids:   # skip already-processed
            continue

        emb_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
        if not os.path.exists(emb_path):
            missing += 1
            continue

        video_emb  = torch.load(emb_path, map_location=DEVICE)
        video_emb  = video_emb / video_emb.norm()

        text_embs  = get_text_embeddings_for_segment(row['masked_caption'])
        similarity = text_embs @ video_emb

        top5_indices = similarity.topk(5).indices.tolist()
        top1_verb    = verb_vocab[top5_indices[0]]
        ground_truth = row['extracted_verb']
        gt_idx       = verb_vocab.index(ground_truth) if ground_truth in verb_vocab else -1

        entry = {
            'segment_id'    : row['segment_id'],
            'subset'        : row['subset'],
            'ground_truth'  : ground_truth,
            'top1_pred'     : top1_verb,
            'top5_preds'    : [verb_vocab[i] for i in top5_indices],
            'top1_correct'  : top1_verb == ground_truth,
            'top5_correct'  : gt_idx in top5_indices,
            'masked_caption': row['masked_caption'],
        }

        results.append(entry)
        ckpt_file.write(json.dumps(entry) + "\n")
        ckpt_file.flush()   # write to disk immediately

print(f"Predicted          : {len(results):,}")
print(f"Missing embeddings : {missing:,}")

In [ ]:
import json
import pandas as pd
from collections import Counter

CKPT_PATH = os.path.join(PATHS['processed'], 'dinov2_zeroshot_fullval_checkpoint.jsonl')

# ── Load ─────────────────────────────────────────────────────────────
results = []
with open(CKPT_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            results.append(json.loads(line))

results_df = pd.DataFrame(results)
total = len(results_df)

# ── Overall ──────────────────────────────────────────────────────────
top1 = results_df['top1_correct'].mean() * 100
top5 = results_df['top5_correct'].mean() * 100

print("=" * 50)
print(f"  DinoTXT Zero-Shot — Full Validation Results")
print("=" * 50)
print(f"  Segments evaluated : {total:,}")
print(f"  Top-1 Accuracy     : {top1:.2f}%")
print(f"  Top-5 Accuracy     : {top5:.2f}%")
print("=" * 50)

# ── Majority class baseline ──────────────────────────────────────────
most_common_verb = results_df['ground_truth'].value_counts().index[0]
baseline = (results_df['ground_truth'] == most_common_verb).mean() * 100
print(f"\n  Majority-class baseline (always predict '{most_common_verb}'): {baseline:.2f}%")

# ── Most common ground truth verbs + their accuracy ──────────────────
print("\nTop-10 most frequent verbs:")
print(f"  {'Verb':<15} {'Count':>6}  {'Top-1':>7}  {'Top-5':>7}")
print(f"  {'-'*15} {'-'*6}  {'-'*7}  {'-'*7}")
for verb, count in results_df['ground_truth'].value_counts().head(10).items():
    sub = results_df[results_df['ground_truth'] == verb]
    t1 = sub['top1_correct'].mean() * 100
    t5 = sub['top5_correct'].mean() * 100
    print(f"  {verb:<15} {count:>6}  {t1:>6.1f}%  {t5:>6.1f}%")

## Results

In [ ]:
# results_df already built in the cell above (val-only). Overall == val-only now,
# so the training/validation split breakdown that used to live here is gone —
# there's no training-subset data in this run by design.
top1_overall = results_df['top1_correct'].mean()
top5_overall = results_df['top5_correct'].mean()
print(f"Val Top-1 : {top1_overall:.4f}  ({top1_overall*100:.2f}%)")
print(f"Val Top-5 : {top5_overall:.4f}  ({top5_overall*100:.2f}%)")
print(f"N         : {len(results_df):,}")

## Per-verb accuracy breakdown

In [ ]:
per_verb = results_df.groupby('ground_truth').agg(
    count      = ('top1_correct', 'count'),
    top1_acc   = ('top1_correct', 'mean'),
    top5_acc   = ('top5_correct', 'mean'),
).sort_values('count', ascending=False)

print("Top 20 most frequent verbs:")
print(per_verb.head(20).to_string())

## Save results

In [ ]:
# Canonical filename — matches what notebooks/scripts/dinov2_zeroshot_fullval_fixed.py
# produces, and what NB5b's (fixed) comparison cells look for.
out_path = os.path.join(PATHS['processed'], 'dinov2_zeroshot_fullval_fixed.csv')
results_df.to_csv(out_path, index=False)
print(f"Results saved → {out_path}")
print(f"Total predictions : {len(results_df):,}")